# Investigation 13: Verify by information


## 1. Setup, Data Loading & Constants


In [ ]:
import pandas as pd
import numpy as np
import os
import ast
import textwrap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from collections import Counter
import itertools
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append(os.path.abspath('../../'))

# --- CONSTANTS ---
NO_SELF = True

MIN_ANT_SUPPORT = 0.03
MIN_JOINT_SUPPORT = 0.01
RESULT_FOLDER_NAME = 'visualization/notebooks/investigation_12_complex_rules.ipynb'

In [2]:

# --- DATA LOADING ---
def get_data_dir():
    current_dir = os.path.abspath(os.getcwd())
    while current_dir != os.path.dirname(current_dir):
        potential_data = os.path.join(current_dir, 'data', 'MIBIGutCsv')
        if os.path.exists(potential_data):
            return potential_data
        current_dir = os.path.dirname(current_dir)
    return '../../data/MIBIGutCsv' 

data_dir = get_data_dir()
df_cells = pd.read_csv(os.path.join(data_dir, 'cell_table.csv'))
df_fovs = pd.read_csv(os.path.join(data_dir, 'fovs_metadata.csv'))

fov_to_size = df_fovs.set_index('FOV')['Size [um]'].to_dict()
def normalize_coords(row):
    size = fov_to_size.get(row['fov'], 800)
    res = 1024 if size == 400 else 2048
    return row['centroid_x'] * (size / res), row['centroid_y'] * (size / res)

df_cells[['x_um', 'y_um']] = df_cells.apply(lambda row: pd.Series(normalize_coords(row)), axis=1)
if 'cell type' in df_cells.columns:
    df_cells['cell_type'] = df_cells['cell type']
else:
    df_cells['cell type'] = df_cells['cell_type']

all_types = sorted(df_cells["cell type"].dropna().astype(str).unique())
try:
    palette = plt.colormaps.get_cmap("tab20b").resampled(max(len(all_types), 1))
except AttributeError:
    palette = plt.cm.get_cmap("tab20b", max(len(all_types), 1))
CELL_COLOR_MAP = {ct: palette(i) for i, ct in enumerate(all_types)}
OTHER_COLOR = (0.5, 0.5, 0.5)

results_dir = os.path.join(os.path.dirname(os.path.dirname(get_data_dir())), 'results', 'full_run', 'weighted_fpgrowth_4_items_no_markers_with_ex_per', 'data')
if not os.path.exists(results_dir):
    results_dir = f'../../results/full_run/{RESULT_FOLDER_NAME}/data'

if not os.path.exists(results_dir):
    for root, dirs, files in os.walk('../../results/'):
        if 'results_CN.csv' in files and 'weighted_fpgrowth' in root:
            results_dir = root
            break

rules_path = os.path.join(results_dir, 'results_CN.csv')
if os.path.exists(rules_path):
    results_orig = pd.read_csv(rules_path)
    print(f"Loaded {len(results_orig)} rules from weighted FP-growth.")
else:
    print(f"File not found: {rules_path}")
    results_orig = pd.DataFrame()



Loaded 275657 rules from weighted FP-growth.


In [3]:
def filter_by_fdr(df, fdr_threshold=0.05):
    if 'FDR' in df.columns:
        return df[df['FDR'] <= fdr_threshold]
    else:
        print("Warning: 'FDR' column not found in DataFrame.")
        return df

In [4]:
def filter_by_lift_improvement(df, lift_improvement_threshold=1.1):
    """
    Filter A: Filter any complex rule where there is a simpler rule (less antecedents, same consequent base types) 
    and the lift isn't improved by x (constant). Matches the pipeline's logic with the length fix.
    """
    if df.empty:
        return df
        
    df_filtered = df.copy()
    
    def extract_base(item):
        return item.replace('_CENTER', '').replace('_NEIGHBOR', '')
        
    df_filtered['len_ant'] = df_filtered['Antecedents'].apply(lambda x: len(ast.literal_eval(x)))
    df_filtered['ant_pure'] = df_filtered['Antecedents'].apply(lambda x: frozenset(extract_base(i) for i in ast.literal_eval(x)))
    df_filtered['con_pure'] = df_filtered['Consequents'].apply(lambda x: frozenset(extract_base(i) for i in ast.literal_eval(x)))
    
    indices_to_drop = set()
    
    for (fov, con_p), group in df_filtered.groupby(['FOV', 'con_pure']):
        sorted_group = group.sort_values(by='len_ant', ascending=True)
        rows = list(sorted_group.itertuples())
        
        for i in range(len(rows)):
            r_complex = rows[i]
            if r_complex.Index in indices_to_drop:
                continue
            for j in range(i):
                r_simple = rows[j]
                
                # The fixed pipeline logic: strictly shorter length + subset pure base cells
                if r_simple.len_ant < r_complex.len_ant and r_simple.ant_pure <= r_complex.ant_pure:
                    is_neg_simple = r_simple.Lift < 1.0
                    is_neg_complex = r_complex.Lift < 1.0
                    
                    if is_neg_simple != is_neg_complex:
                        continue
                        
                    if is_neg_complex:
                        if r_complex.Lift >= r_simple.Lift / lift_improvement_threshold:
                            indices_to_drop.add(r_complex.Index)
                            break
                    else:
                        if r_complex.Lift <= r_simple.Lift * lift_improvement_threshold:
                            indices_to_drop.add(r_complex.Index)
                            break
                            
    result_df = df_filtered.drop(index=list(indices_to_drop)).drop(columns=['len_ant', 'ant_pure', 'con_pure'])
    print(f"Filter A: Filtered {len(indices_to_drop)} rules. Remaining: {len(result_df)}")
    return result_df


In [5]:
def filter_by_all_simple_exist(df):
    """
    Filter B: Filter any complex rule that all of its antecedents have a simpler rule (1-item antecedent)
    with the same consequent, never mind what's the improvement in lift.
    """
    if df.empty:
        return df
        
    df_filtered = df.copy()
    df_filtered['ant_set'] = df_filtered['Antecedents'].apply(lambda x: frozenset(ast.literal_eval(x)))
    df_filtered['con_set'] = df_filtered['Consequents'].apply(lambda x: frozenset(ast.literal_eval(x)))
    
    indices_to_drop = set()
    
    for fov, group in df_filtered.groupby('FOV'):
        complex_rules = group[group['ant_set'].apply(len) > 1]
        if complex_rules.empty:
            continue
            
        # create a lookup for simple rules in this FOV: (ant_item, con_set) -> exists
        simple_rules = group[group['ant_set'].apply(len) == 1]
        simple_lookup = set()
        for r_simple in simple_rules.itertuples():
            ant_item = list(r_simple.ant_set)[0]
            simple_lookup.add((ant_item, r_simple.con_set))
            
        for r_complex in complex_rules.itertuples():
            # check if ALL items in complex antecedent have a simple rule with the same consequent
            all_exist = True
            for ant_item in r_complex.ant_set:
                if (ant_item, r_complex.con_set) not in simple_lookup:
                    all_exist = False
                    break
            
            if all_exist:
                indices_to_drop.add(r_complex.Index)
                
    result_df = df_filtered.drop(index=list(indices_to_drop)).drop(columns=['ant_set', 'con_set'])
    print(f"Filter B: Filtered {len(indices_to_drop)} rules. Remaining: {len(result_df)}")
    return result_df


In [6]:
def filter_by_center_biggest_support(df, df_cells):
    """
    Filter C: Remove complex rules if the center of the rule isn't the cell 
    with the biggest support of all antecedent types in the FOV.
    Support here is calculated as the cell count in the FOV.
    """
    if df.empty:
        return df
        
    df_filtered = df.copy()
    
    # Precompute cell counts per FOV
    fov_cell_counts = df_cells.groupby(['fov', 'cell type']).size().to_dict()
    
    def get_center_type(rule_str):
        for item in ast.literal_eval(rule_str):
            if '_CENTER' in item:
                return item.replace('_CENTER', '')
        return None

    def get_base_types(rule_str):
        return [item.replace('_CENTER', '').replace('_NEIGHBOR', '') for item in ast.literal_eval(rule_str)]
        
    indices_to_drop = set()
    
    for row in df_filtered.itertuples():
        ant_list = ast.literal_eval(row.Antecedents)
        if len(ant_list) <= 1:
            continue
            
        center_type = get_center_type(row.Antecedents)
        if not center_type:
            continue
            
        base_types = get_base_types(row.Antecedents)
        
        # Get counts for all base types in the antecedent
        counts = {bt: fov_cell_counts.get((row.FOV, bt), 0) for bt in base_types}
        max_count = max(counts.values())
        
        # If the center type does not have the strictly or equal maximum count, drop it
        if counts.get(center_type, 0) < max_count:
            indices_to_drop.add(row.Index)
            
    result_df = df_filtered.drop(index=list(indices_to_drop))
    print(f"Filter C: Filtered {len(indices_to_drop)} rules. Remaining: {len(result_df)}")
    return result_df


In [7]:
def filter_by_min_ant_support(df, min_support=MIN_ANT_SUPPORT):
    if df.empty: return df
    df_filtered = df.copy()
    # Support is joint support. Confidence = joint_support / ant_support => ant_support = Support / Confidence
    df_filtered['ant_support'] = df_filtered['Support'] / df_filtered['Confidence'].clip(lower=1e-9)
    df_filtered = df_filtered[df_filtered['ant_support'] >= min_support]
    return df_filtered.drop(columns=['ant_support'])


In [8]:
def filter_by_min_joint_support(df, min_support=MIN_JOINT_SUPPORT):
    if df.empty: return df
    return df[df['Support'] >= min_support]


In [9]:
# Call the filters one by one
print(f"Starting with {len(results_orig)} rules")

results = results_orig.copy()
results = filter_by_fdr(results, fdr_threshold=0.05)
# Apply Filter A
results = filter_by_lift_improvement(results, lift_improvement_threshold=1.3)

# Apply Filter B
results = filter_by_all_simple_exist(results)

# Apply Filter C
results = filter_by_center_biggest_support(results, df_cells)


print(f'Before D: {len(results)}')
results = filter_by_min_ant_support(results, MIN_ANT_SUPPORT)
print(f'After D: {len(results)}')

print(f'Before E: {len(results)}')
results = filter_by_min_joint_support(results, MIN_JOINT_SUPPORT)
print(f'After E: {len(results)}')


Starting with 275657 rules
Filter A: Filtered 21678 rules. Remaining: 74550
Filter B: Filtered 0 rules. Remaining: 74550
Filter C: Filtered 4880 rules. Remaining: 69670
Before D: 69670
After D: 53292
Before E: 53292
After E: 21583


# P Models:

In [14]:
def precompute_fov_neighbors(df_fov, radius=30.0):
    """
    Builds a spatial neighborhood graph for a single FOV exactly once.
    
    Args:
        df_fov (pd.DataFrame): Dataframe containing cells for a single FOV.
        radius (float): The spatial radius for the neighborhood.
        
    Returns:
        dict: A mapping from the cell's dataframe index to an array of 
              its neighbor dataframe indices.
    """
    coords = df_fov[['x_um', 'y_um']].values
    nn = NearestNeighbors(radius=radius)
    nn.fit(coords)
    
    # Get neighbors for ALL cells in the FOV at once
    all_neighbors_indices = nn.radius_neighbors(coords, return_distance=False, sort_results=False)
    
    neighborhood_graph = {}
    
    for i, cell_idx in enumerate(df_fov.index):
        # Convert integer positions back to actual dataframe indices
        neighbor_idx = df_fov.index[all_neighbors_indices[i]]
        
        if NO_SELF:
            # Remove the cell itself from its own neighbor list
            neighbor_idx = neighbor_idx[neighbor_idx != cell_idx]
            
        neighborhood_graph[cell_idx] = neighbor_idx
        
    return neighborhood_graph

In [15]:
def extract_patch_states(df_fov, neighborhood_graph, center_type, neighbor_types):
    """
    Extracts the boolean state of specified neighbor cell types around each center cell,
    using a precomputed neighborhood graph for O(1) lookups.
    
    Args:
        df_fov (pd.DataFrame): Dataframe containing cells for a single FOV.
        neighborhood_graph (dict): The precomputed neighbors from precompute_fov_neighbors().
        center_type (str): The cell type acting as the patch center.
        neighbor_types (list of str): The cell types we are looking for.
        
    Returns:
        pd.DataFrame: A matrix of boolean indicators (0 or 1) for each neighbor_type.
    """
    # Isolate only the center cells
    centers = df_fov[df_fov['cell_type'] == center_type]
    
    if centers.empty:
        return pd.DataFrame(columns=neighbor_types)
    
    states_list = []
    
    # Loop only over the center cells we care about
    for center_idx in centers.index:
        # O(1) lookup to get neighbor indices
        neighbor_indices = neighborhood_graph[center_idx]
        
        # Get the actual neighbor cells
        neighborhood_cells = df_fov.loc[neighbor_indices]
        
        present_types = set(neighborhood_cells['cell_type'].unique())
        state_vector = {ntype: int(ntype in present_types) for ntype in neighbor_types}
        states_list.append(state_vector)
        
    return pd.DataFrame(states_list)

In [16]:
def calculate_smoothed_probabilities(df_states, alpha=0.01):
    """
    Calculates the empirical probability distribution of states 
    using Laplace Smoothing to prevent zero probabilities.
    """
    N = len(df_states)
    if N == 0:
        return {}
        
    num_neighbors = len(df_states.columns)
    
    # Calculate the size of the state space (|M|)
    # For k binary neighbor columns, there are exactly 2^k possible combinations
    M_size = 2 ** num_neighbors
    
    # Convert each row to a tuple (e.g., (1,0)) to make it hashable, 
    # then count exactly how many times each spatial motif appeared in the tissue
    state_counts = Counter(tuple(row) for row in df_states.itertuples(index=False, name=None))
    
    # Generate the full theoretical state space (including motifs never observed)
    # Cartesian product of [0, 1] repeated 'num_neighbors' times
    all_possible_states = list(itertools.product([0, 1], repeat=num_neighbors))
    
    # Denominator for Laplace Smoothing: N + alpha * |M|
    denominator = N + (alpha * M_size)
    p_dist = {}
    
    for state in all_possible_states:
        # Retrieve the empirical count, defaulting to 0 if the motif never appeared
        count_m = state_counts.get(state, 0)
        
        # Apply the smoothing formula to ensure strict positivity
        p_dist[state] = (count_m + alpha) / denominator
        
    return p_dist

In [17]:
def create_macro_null_model(df_fov, grid_size=100.0):
    """
    Creates the P_macro null model by locally shuffling cell types within spatial grid bins.
    """
    df_shuffled = df_fov.copy()
    
    # 1. Map continuous coordinates into discrete bins (e.g., 100um blocks)
    bin_x = (df_shuffled['x_um'] // grid_size).astype(int)
    bin_y = (df_shuffled['y_um'] // grid_size).astype(int)
    
    # 2. Create a unified string identifier for each spatial block (e.g., "3_5")
    df_shuffled['grid_bin'] = bin_x.astype(str) + "_" + bin_y.astype(str)
    
    # 3. Group by the spatial block and randomly shuffle ONLY the 'cell_type' column
    df_shuffled['cell_type'] = df_shuffled.groupby('grid_bin')['cell_type'].transform(np.random.permutation)
    
    # 4. Remove the temporary grouping column to perfectly match the original dataframe structure
    df_shuffled = df_shuffled.drop(columns=['grid_bin'])
    
    return df_shuffled

In [19]:
def apply_ipf_for_rule(p_macro, p_real, rule_states, max_iters=100, tol=1e-5):
    """
    Adjusts P_macro to match the empirical probability of a specific rule from P_real 
    using Iterative Proportional Fitting (IPF).
    (Creating Pk)
    """
    # Start with the baseline macro model
    p_k = p_macro.copy()
    
    # Calculate the exact target probability of this rule in reality
    target_prob = sum(p_real[state] for state in rule_states)
    
    for i in range(max_iters):
        # Calculate the current probability of the rule in the running model
        current_prob = sum(p_k[state] for state in rule_states)
        
        # Prevent division by zero if the rule is completely absent
        if current_prob == 0:
            break
            
        # The IPF update ratio: Target / Current
        ratio = target_prob / current_prob
        
        # Apply the update ONLY to the states that satisfy the rule
        for state in rule_states:
            p_k[state] *= ratio
            
        # Normalize the entire distribution so the sum is exactly 1.0 again
        total_sum = sum(p_k.values())
        for state in p_k:
            p_k[state] /= total_sum
            
        # Check for convergence: if the model matches reality, stop the loop
        new_prob = sum(p_k[state] for state in rule_states)
        if abs(new_prob - target_prob) < tol:
            break
            
    return p_k

In [ ]:
import math

def calculate_kl_divergence(p_real, p_model):
    """
    Calculates the Kullback-Leibler Divergence D_KL(P_real || P_model) in bits.
    """
    kl_div = 0.0
    for state in p_real:
        # We only calculate for states that actually occur in reality
        if p_real[state] > 0 and p_model[state] > 0:
            kl_div += p_real[state] * math.log2(p_real[state] / p_model[state])
    
    return kl_div



Calculate IG

    IMPORTANT NOTE: 
    Because both P_macro and P_k share the exact same global anatomy distribution, 
    the macro information cancels out during the subtraction. The resulting Information Gain 
    strictly represents the MICRO information (local cell-to-cell interactions) 
    that the specific rule adds to the system.


In [20]:
def calculate_information_gain(p_real, p_macro, p_k):
    """
    Calculates the Information Gain (IG) of a specific spatial rule.
    """
    # Step 1: Baseline Information Loss (Reality vs. Macro null model)
    i_baseline = calculate_kl_divergence(p_real, p_macro)
    
    # Step 2: Remaining Information Loss (Reality vs. IPF micro-rule model)
    i_loss = calculate_kl_divergence(p_real, p_k)
    
    # Step 3: Information Gain (Micro information successfully explained by the rule)
    ig_k = i_baseline - i_loss
    
    return ig_k


## IPF implementation (P integrated)

In [1]:
def build_integrated_model(p_macro, p_real, list_of_rules, max_iters=100, tol=1e-5):
    """
    Builds the integrated model (P_integrated) by applying IPF to multiple rules simultaneously.
    This calculates the combined spatial structure explained by a group of rules.
    """
    p_integrated = p_macro.copy()
    
    # Pre-calculate the target empirical probabilities for all rules from reality
    target_probs = [sum(p_real[state] for state in rule_states) for rule_states in list_of_rules]
    
    for i in range(max_iters):
        max_diff = 0.0
        
        # Sequentially update the model for each rule in the list
        for rule_idx, rule_states in enumerate(list_of_rules):
            target_prob = target_probs[rule_idx]
            current_prob = sum(p_integrated[state] for state in rule_states)
            
            # Avoid division by zero if a rule's probability drops to absolute zero
            if current_prob == 0:
                continue
                
            # The IPF update ratio: Target / Current
            ratio = target_prob / current_prob
            
            # Apply the update only to the states that satisfy the current rule
            for state in rule_states:
                p_integrated[state] *= ratio
                
            # Normalize the entire distribution so the sum is exactly 1.0
            total_sum = sum(p_integrated.values())
            for state in p_integrated:
                p_integrated[state] /= total_sum
                
            # Track the largest deviation to check for global convergence
            new_prob = sum(p_integrated[state] for state in rule_states)
            diff = abs(new_prob - target_prob)
            if diff > max_diff:
                max_diff = diff
                
        # If the largest deviation across ALL rules is below the tolerance, we converged
        if max_diff < tol:
            break
            
    return p_integrated

## Cumulative Information Pipeline (The Elbow Curve Data Generator)

In [2]:
def evaluate_cumulative_information(p_macro, p_real, ranked_rules, step=1):
    """
    Evaluates the cumulative Information Gain as ranked rules are incrementally added.
    Returns data points (x_values, y_values) suitable for plotting the 'Elbow Curve'.
    """
    x_values = []
    y_values = []
    
    # Calculate the baseline information loss once (Reality vs. Macro null model)
    baseline_loss = calculate_kl_divergence(p_real, p_macro)
    
    total_rules = len(ranked_rules)
    
    # Incrementally add rules based on the defined step size
    for i in range(1, total_rules + 1, step):
        # Take the top 'i' rules from the ranked list
        current_rules = ranked_rules[:i]
        
        # Build the integrated model with the current subset of rules
        p_integrated = build_integrated_model(p_macro, p_real, current_rules)
        
        # Calculate the remaining information loss
        current_loss = calculate_kl_divergence(p_real, p_integrated)
        
        # Calculate the cumulative Information Gain (in bits)
        cumulative_ig = baseline_loss - current_loss
        
        # Store the results for the graph
        x_values.append(i)
        y_values.append(cumulative_ig)
        
    # Ensure the final point includes absolutely all rules, 
    # just in case the loop step skipped the exact final index.
    if len(x_values) == 0 or x_values[-1] != total_rules:
        p_integrated = build_integrated_model(p_macro, p_real, ranked_rules)
        current_loss = calculate_kl_divergence(p_real, p_integrated)
        cumulative_ig = baseline_loss - current_loss
        
        x_values.append(total_rules)
        y_values.append(cumulative_ig)
        
    return x_values, y_values

### Spatial Cross-Validation (Rule Transfer to Test FOV)

In [ ]:
def validate_rules_on_test_region(p_macro_test, p_real_test, golden_rules):
    """
    Validates the extracted rules on a new Field of View (FOV).
    Calculates how much spatial information the training rules can explain in a new anatomical context.
    """
    # Calculate the baseline information loss in the new test region
    baseline_loss_test = calculate_kl_divergence(p_real_test, p_macro_test)
    
    # Apply the golden rules from the training region to the test region's macro anatomy
    p_integrated_test = build_integrated_model(p_macro_test, p_real_test, golden_rules)
    
    # Calculate the remaining information loss in the test region
    integrated_loss_test = calculate_kl_divergence(p_real_test, p_integrated_test)
    
    # Calculate the confirmed Information Gain on the unseen tissue
    validation_ig = baseline_loss_test - integrated_loss_test
    
    return validation_ig